# 📊 Data Quality Observability & Intelligent Remediation
**Autonomous AI-Agent Platform for Enterprise Data Health**

▶️ Run the cell below to launch the platform. This will:
1.  Clone the repository
2.  Install dependencies
3.  Setup a public URL for access

--- 
### 🔑 Optional: Enable Email Alerts
Add credentials via **Colab Secrets** (🔑 icon in the left sidebar) before running:
- `SENDGRID_API_KEY` 
- `DQ_ALERT_RECIPIENTS` 
- `NGROK_AUTH_TOKEN` (Optional: for a more stable tunnel)

In [ ]:
# @title 🚀 Launch Platform
import os, subprocess, threading, time, sys
from google.colab import userdata

print("🔄 Starting setup...")

# 1. Clone repository
REPO_URL = "https://github.com/Teja-Jan/Data-Quality-Observability-Intelligent-Remediation.git"
REPO_DIR = "Data-Quality-Observability-Intelligent-Remediation"
BRANCH   = "Data-Quality-Observability-and-Intelligent-Remediation"

if not os.path.exists(REPO_DIR):
    print("📥 Cloning repository...")
    !git clone -b $BRANCH $REPO_URL
else:
    print("✅ Repo already exists. Updating...")
    %cd $REPO_DIR
    !git pull
    %cd ..

%cd $REPO_DIR

# 2. Install dependencies
print("📦 Installing dependencies (may take a minute)...")
!pip install -q -r requirements.txt
!pip install -q pyngrok
!npm install -g localtunnel > /dev/null 2>&1

# 3. Set up environment
print("🔑 Configuring environment...")
def get_secret(key):
    try:
        return userdata.get(key)
    except:
        return ""

SG_KEY = get_secret('SENDGRID_API_KEY')
SG_TO  = get_secret('DQ_ALERT_RECIPIENTS')
NGROK  = get_secret('NGROK_AUTH_TOKEN')

with open(".env", "w") as f:
    f.write(f"EMAIL_PROVIDER={'sendgrid' if SG_KEY else 'smtp'}\n")
    f.write(f"SENDGRID_API_KEY={SG_KEY}\n")
    f.write(f"DQ_ALERT_RECIPIENTS={SG_TO}\n")
    f.write("APP_ENV=colab\n")
    f.write("LOG_LEVEL=INFO\n")
    f.write("DB_PATH=src/db/dq_metadata.db\n")

# 4. Start Ollama in background
print("🤖 Initializing AI engine (Ollama)...")
def setup_ollama():
    os.system("curl -fsSL https://ollama.com/install.sh | sh")
    os.system("ollama serve > /dev/null 2>&1 &")
    time.sleep(5)
    os.system("ollama pull llama3")

threading.Thread(target=setup_ollama, daemon=True).start()

# 5. Launch Streamlit
print("🚀 Launching application...")
PORT = 8501

# Cleanup existing process and set PYTHONPATH
os.system(f"fuser -k {PORT}/tcp > /dev/null 2>&1")
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.getcwd()}/src"

get_ipython().system_raw(f'streamlit run src/app.py --server.port {PORT} &')

# 6. Tunnelling
time.sleep(8)
print("\n" + "="*40)
print("🌐 ACCESS YOUR APPLICATION")
print("="*40)

# Try ngrok first if token exists
tunnel_url = ""
if NGROK:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK)
    tunnel_url = ngrok.connect(PORT).public_url
    print(f"✅ Ngrok Tunnel: {tunnel_url}")
else:
    # Fallback to localtunnel
    import urllib
    print("ℹ️ No ngrok token found. Using localtunnel fallback.")
    print("🔑 Your Endpoint IP (copy this):", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
    print("🔗 Click the link below and paste the IP if prompted:")
    get_ipython().system_raw(f'lt --port {PORT} > tunnel.txt 2>&1 &')
    time.sleep(5)
    if os.path.exists("tunnel.txt"):
        with open("tunnel.txt", "r") as f:
            print(f.read())

print("="*40)
print("ℹ️ Note: If the app doesn't load, wait 10 seconds and refresh.")